# Tema 13 — Reconocimiento facial y verificación biométrica

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-07/Tema-13/Tema_13.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

En este notebook construimos un flujo completo de **verificación de identidad por rostro**: detectamos la cara en una imagen (**MTCNN**), la convertimos en un **vector biométrico** de 512 dimensiones (**FaceNet**) y comparamos dos rostros con la **distancia coseno** para decidir si pertenecen a la misma persona.

> 💡 Si lo abres en Colab, ejecuta primero la celda **Setup para Google Colab** (instala `mtcnn` y `keras-facenet`) y sube tus imágenes de prueba (`foto_pasaporte.png`, `foto_selfie.png`). En entornos locales esa celda no hace nada y puedes saltarla.

In [ ]:
# === Setup para Google Colab ===
# Esta celda instala las dependencias que NO vienen preinstaladas en Colab.
# En entornos locales (VS Code / Jupyter) se ignora; si ya las tienes instaladas
# también puedes saltarla.
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # mtcnn -> detección de rostros | keras-facenet -> embeddings biométricos
    !pip install -q mtcnn keras-facenet
    print("Setup de Colab completado. Reinicia el entorno solo si se te solicita.")
else:
    print("Entorno local detectado, no se requiere setup adicional.")

**Detección y preprocesamiento facial**

### ¿Qué hace este código?

Define la **etapa de detección y preprocesamiento** del rostro:

1. Carga el detector **MTCNN**, una red especializada en encontrar caras dentro de una imagen.
2. La función `detectar_rostro` lee la imagen con **OpenCV** y la convierte de BGR a **RGB**.
3. MTCNN escanea la imagen y devuelve el **cuadro delimitador** (*bounding box*) de la cara detectada.
4. Se **recorta** solo la región del rostro (eliminando el fondo) y se **redimensiona a 160×160 px**, el tamaño que requiere la red FaceNet.

In [ ]:
#pip install mtcnn[tensorflow]
#pip install opencv-python
# Importar librerías necesarias
import cv2
from mtcnn import MTCNN  # Algoritmo popular para detección de rostros

# Cargar el detector
detector = MTCNN()

def detectar_rostro(ruta_imagen):
    # Cargar la imagen visual
    img = cv2.cvtColor(cv2.imread(ruta_imagen), cv2.COLOR_BGR2RGB)

    # ETAPA 1: DETECCIÓN
    # El algoritmo escanea la imagen buscando patrones geométricos (ojos, nariz)
    # Resultados: Coordenadas del cuadro delimitador (bounding box)
    resultados = detector.detect_faces(img)

    if resultados:
        x, y, ancho, alto = resultados[0]['box']
        # Recortar solo la cara (eliminamos el fondo/ruido)
        cara = img[y:y+alto, x:x+ancho]
        # Redimensionar al tamaño que requiere la red neuronal (ej. 160x160 px)
        cara_procesada = cv2.resize(cara, (160, 160))
        return cara_procesada, img, x, y, ancho, alto
    else:
        return None, img, x, y, ancho, alto

**Imágenes de prueba (descarga automática)**

### ¿Qué hace este código?

Las celdas siguientes necesitan dos imágenes (`foto_pasaporte.png` y `foto_selfie.png`). Si **no existen** en el entorno (típico en **Google Colab**), esta celda las **descarga automáticamente** desde un repositorio público de ejemplos de rostros:

- `foto_pasaporte.png` y `foto_selfie.png` son **dos fotos de la misma persona**, por lo que la verificación final debe dar **ACCESO CONCEDIDO**.
- También se descarga `otra_persona.png` (una persona distinta) por si quieres probar el caso **ACCESO DENEGADO**: úsala como selfie en la celda de embeddings.
- Si subes tus **propias imágenes** con esos nombres, la celda las respeta y **no las sobrescribe**.

In [ ]:
# Descarga imágenes de prueba SOLO si no existen ya en el entorno (p. ej. en Colab).
# Fuente: ejemplos públicos del repositorio ageitgey/face_recognition.
import os
import urllib.request

IMAGENES_PRUEBA = {
    # Misma persona en dos fotos distintas -> la verificación debe dar ACCESO CONCEDIDO.
    "foto_pasaporte.png": "https://raw.githubusercontent.com/ageitgey/face_recognition/master/examples/obama.jpg",
    "foto_selfie.png":    "https://raw.githubusercontent.com/ageitgey/face_recognition/master/examples/obama2.jpg",
    # Persona distinta (opcional): úsala como selfie para probar el caso ACCESO DENEGADO.
    "otra_persona.png":   "https://raw.githubusercontent.com/ageitgey/face_recognition/master/examples/biden.jpg",
}

for nombre, url in IMAGENES_PRUEBA.items():
    if os.path.exists(nombre):
        print(f"[=] '{nombre}' ya existe, se conserva (no se sobrescribe).")
    else:
        urllib.request.urlretrieve(url, nombre)
        print(f"[+] Descargada '{nombre}'.")

**Detección de rostros (bounding box) y recortes (crops)**

### ¿Qué hace este código?

**Visualiza** el resultado de la detección sobre dos imágenes (`foto_selfie.png` y `foto_pasaporte.png`):

1. Para cada imagen llama a `detectar_rostro` y obtiene el recorte y las coordenadas del rostro.
2. En la columna izquierda dibuja el **rectángulo verde** sobre la imagen original para mostrar *dónde* se detectó la cara.
3. En la columna derecha muestra el **recorte** (*crop*) del rostro ya aislado.
4. Usa una cuadrícula de **Matplotlib** (2×2) para comparar visualmente original vs. recorte de cada foto.

In [ ]:
import matplotlib.pyplot as plt
archivos = ["foto_pasaporte.png", "foto_selfie.png"]
titulos_filas = ["Selfie", "Documento"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, archivo in enumerate(archivos):
    # Obtener datos de tu función
    rostro, img_orig, x, y, w, h = detectar_rostro(archivo)

    if rostro is not None:
        # --- A. IMAGEN CON RECTÁNGULO (Contexto) ---
        img_con_box = img_orig.copy()
        # Dibujar rectángulo VERDE
        cv2.rectangle(img_con_box, (x, y), (x+w, y+h), (0, 255, 0), 4)

        axes[i, 0].imshow(img_con_box, aspect='equal') # aspect='equal' evita deformación
        axes[i, 0].set_title(f"{titulos_filas[i]} (Original)", fontsize=10)
        axes[i, 0].axis('off')

        # --- B. RECORTE (Crop) ---
        axes[i, 1].imshow(rostro, aspect='equal')
        axes[i, 1].set_title(f"Recorte detectado ({w}x{h} px)", fontsize=10)

        # Quitamos los ejes pero mantenemos el marco para que se vea el límite del crop
        axes[i, 1].axis('off')

    else:
        axes[i, 0].text(0.5, 0.5, "No detectado", ha='center')
        axes[i, 1].text(0.5, 0.5, "N/A", ha='center')

plt.tight_layout()
plt.show()

**Generación de vectores biométricos**

### ¿Qué hace este código?

Convierte cada rostro en su **firma biométrica** (*embedding*):

1. Carga el modelo preentrenado **FaceNet** (`keras-facenet`), que ya aprendió a representar caras como vectores numéricos.
2. La función `calcular_embedding` recibe el rostro de 160×160 px y genera un **vector de 512 dimensiones** que resume las características únicas de esa cara.
3. Se calculan los *embeddings* del **pasaporte** y de la **selfie**.
4. Se imprime la dimensión del vector y sus primeros valores como ejemplo de la "huella" matemática del rostro.

In [ ]:
#pip install keras-facenet numpy
#pip install pip-system-certs
import numpy as np
from keras_facenet import FaceNet

# 1. Cargar el modelo FaceNet pre-entrenado
# Este modelo ya "sabe" cómo convertir caras en números gracias a un entrenamiento previo masivo
embedder = FaceNet()

def calcular_embedding(rostro_procesado):
    """
    Recibe: Imagen del rostro (160x160 px).
    Devuelve: Vector numérico (Embedding) de 512 dimensiones.
    """
    if rostro_procesado is None:
        return None
    # Preprocesamiento: FaceNet espera un lote (batch) de imágenes
    muestras = np.expand_dims(rostro_procesado, axis=0)

    # CÁLCULO DEL EMBEDDING
    # La red neuronal recorre la imagen y genera el vector de características
    embedding = embedder.embeddings(muestras)

    # Devolvemos solo el primer vector (porque solo pasamos una imagen)
    return embedding[0]

# --- EJECUCIÓN (Usando las salidas del código anterior) ---
rostro_pasaporte, img_orig, x, y, w, h = detectar_rostro("foto_pasaporte.png")
rostro_selfie, img_orig, x, y, w, h = detectar_rostro("foto_selfie.png")
vector_pasaporte = calcular_embedding(rostro_pasaporte)
vector_selfie = calcular_embedding(rostro_selfie)

# Ejemplo de visualización de los datos
print(f"Dimensión del vector: {len(vector_pasaporte)}")
print(f"Primeros 5 valores (Firma Biométrica): {vector_pasaporte[:5]}")

**Algoritmo de verificación**

### ¿Qué hace este código?

Realiza la **decisión final** de verificación comparando los dos vectores:

1. Calcula la **distancia coseno** entre el *embedding* del pasaporte y el de la selfie. Un valor cercano a **0** indica rostros casi idénticos; cercano a **1**, rostros distintos.
2. Compara la distancia contra un **umbral** (`0.40`, un estándar común para FaceNet).
3. Si la distancia es **menor** al umbral → **ACCESO CONCEDIDO** (misma persona); si no → **ACCESO DENEGADO**.
4. Imprime un reporte legible con la distancia, el umbral de seguridad y el veredicto.

In [ ]:
from scipy.spatial.distance import cosine

# ==========================================
# ETAPA 3: COMPARACIÓN Y VERIFICACIÓN
# ==========================================

def comparar_embeddings(vector_referencia, vector_prueba, umbral=0.40):
    """
    Compara dos vectores biométricos usando la Distancia Coseno.

    Parámetros:
        vector_referencia: Embedding del pasaporte.
        vector_prueba: Embedding de la selfie.
        umbral (threshold): Límite máximo de diferencia permitida.
                            (0.40 es un estándar común para FaceNet).
    """
    # La distancia coseno mide el ángulo entre dos vectores.
    # Valor cercano a 0: Los vectores son casi idénticos (Misma Persona).
    # Valor cercano a 1: Los vectores son muy diferentes (Personas Distintas).
    distancia = cosine(vector_referencia, vector_prueba)

    # Decisión Lógica
    es_misma_persona = distancia < umbral

    return distancia, es_misma_persona

# --- EJEMPLO DE USO (Continuación del flujo) ---
distancia_calculada, resultado = comparar_embeddings(vector_pasaporte, vector_selfie)

# Visualización del Resultado para el Usuario
print(f"--- REPORTE DE VERIFICACIÓN ---")
print(f"Distancia Biométrica: {distancia_calculada}") # Ejemplo de valor real
print(f"Umbral de Seguridad:  0.40")

if resultado: # Simulación de resultado positivo (resultado == True)
    print("\n[+] ACCESO CONCEDIDO: Identidad Verificada.")
    print("    Las firmas matemáticas coinciden.")
else:
    print("\n[-] ACCESO DENEGADO: Rostros no coinciden.")